# RAFT Dataset Generation: CUDA to HIP with Chain-of-Thought

**Notebook:** `FunctionalLoraTest.ipynb`  
**Objective:** Construct a high-quality, reasoning-aware dataset for CUDA-to-HIP migration.  
**Methodology:** RAFT (Reasoning-Aware Fine-Tuning). We combine automated translation via `HIPIFY` with LLM-generated Chain-of-Thought (CoT) reasoning based on official ROCm documentation.

### Why RAFT?
Standard fine-tuning often leads to models that memorize mappings without understanding. By providing **Gold Documentation** and **Distractors** during the generation phase, we train the model to:
1. Identify relevant ROCm APIs.
2. Justify the migration steps.
3. Filter out irrelevant information.

## 1. Massive Repository Ingestion
To ensure the model handles real-world code, we clone a diverse set of high-performance computing (HPC) libraries, including NVIDIA's core samples, `CUTLASS`, `Thrust`, and `Megatron-LM`. This provides thousands of `.cu` files across various complexity levels.

In [1]:
!git clone https://github.com/NVIDIA/cuda-samples
!git clone https://github.com/NVIDIA/cutlass
!git clone https://github.com/NVIDIA/thrust
!git clone https://github.com/moderngpu/moderngpu

!git clone https://github.com/ROCm/HIPIFY

Cloning into 'cuda-samples'...
remote: Enumerating objects: 31311, done.
remote: Counting objects: 100% (7113/7113), done.
remote: Compressing objects: 100% (246/246), done.
remote: Total 31311 (delta 6911), reused 6867 (delta 6867), pack-reused 24198 (from 1)
Receiving objects: 100% (31311/31311), 136.91 MiB | 30.38 MiB/s, done.
Resolving deltas: 100% (27591/27591), done.
Updating files: 100% (2054/2054), done.
Cloning into 'cutlass'...
remote: Enumerating objects: 42752, done.
remote: Counting objects: 100% (393/393), done.
remote: Compressing objects: 100% (276/276), done.
remote: Total 42752 (delta 248), reused 117 (delta 117), pack-reused 42359 (from 4)
Receiving objects: 100% (42752/42752), 66.45 MiB | 27.26 MiB/s, done.
Resolving deltas: 100% (32297/32297), done.
Cloning into 'thrust'...
remote: Enumerating objects: 50516, done.
remote: Counting objects: 100% (234/234), done.
remote: Compressing objects: 100% (140/140), done.
remote: Total 50516 (delta 98), reused 199 (delta 85)

In [2]:
!git clone https://github.com/NVIDIA/cub
!git clone https://github.com/NVIDIA/apex
!git clone https://github.com/NVIDIA/Megatron-LM
!git clone https://github.com/facebookresearch/xformers
!git clone https://github.com/NVIDIA/cuda-samples

Cloning into 'cub'...
remote: Enumerating objects: 33392, done.
remote: Counting objects: 100% (247/247), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 33392 (delta 209), reused 184 (delta 184), pack-reused 33145 (from 4)
Receiving objects: 100% (33392/33392), 18.00 MiB | 27.47 MiB/s, done.
Resolving deltas: 100% (27972/27972), done.
Cloning into 'apex'...
remote: Enumerating objects: 13204, done.
remote: Counting objects: 100% (710/710), done.
remote: Compressing objects: 100% (500/500), done.
remote: Total 13204 (delta 406), reused 211 (delta 210), pack-reused 12494 (from 4)
Receiving objects: 100% (13204/13204), 16.69 MiB | 26.88 MiB/s, done.
Resolving deltas: 100% (8930/8930), done.
Cloning into 'Megatron-LM'...
remote: Enumerating objects: 106335, done.
remote: Counting objects: 100% (267/267), done.
remote: Compressing objects: 100% (141/141), done.
remote: Total 106335 (delta 194), reused 132 (delta 126), pack-reused 106068 (from 2)
Receiving objects: 100%

In [3]:
!git clone https://github.com/NVIDIA/cccl
!git clone https://github.com/NVIDIA/CUDALibrarySamples
!git clone https://github.com/NVIDIA/nvbench
!git clone https://github.com/NVIDIA/warp
!git clone https://github.com/NVIDIA/cuCollections

!git clone https://github.com/ShlokVFX/100-days-cuda
!git clone https://github.com/abaksy/cuda-examples
!git clone https://github.com/drkennetz/cuda_examples
!git clone https://github.com/ashokyannam/GPU_Acceleration_Using_CUDA_C_CPP
!git clone https://github.com/udacity/cs344

!git clone https://github.com/clu0/unet.cu
!git clone https://github.com/mobiusml/gemlite
!git clone https://github.com/Dao-AILab/flash-attention
!git clone https://github.com/NVIDIA/FasterTransformer
!git clone https://github.com/NVIDIA/TransformerEngine

!git clone https://github.com/NVIDIA/amgx
!git clone https://github.com/NVIDIA/cuda-quantum
!git clone https://github.com/rapidsai/cudf
!git clone https://github.com/rapidsai/cuml

Cloning into 'cccl'...
remote: Enumerating objects: 317076, done.
remote: Counting objects: 100% (2544/2544), done.
remote: Compressing objects: 100% (465/465), done.
remote: Total 317076 (delta 2321), reused 2098 (delta 2075), pack-reused 314532 (from 3)
Receiving objects: 100% (317076/317076), 125.07 MiB | 25.77 MiB/s, done.
Resolving deltas: 100% (245197/245197), done.
Cloning into 'CUDALibrarySamples'...
remote: Enumerating objects: 9106, done.
remote: Counting objects: 100% (2642/2642), done.
remote: Compressing objects: 100% (705/705), done.
remote: Total 9106 (delta 2109), reused 2198 (delta 1909), pack-reused 6464 (from 2)
Receiving objects: 100% (9106/9106), 40.04 MiB | 31.54 MiB/s, done.
Resolving deltas: 100% (5992/5992), done.
Cloning into 'nvbench'...
remote: Enumerating objects: 6055, done.
remote: Counting objects: 100% (3261/3261), done.
remote: Compressing objects: 100% (808/808), done.
remote: Total 6055 (delta 2813), reused 2483 (delta 2449), pack-reused 2794 (from 3

In [4]:
import os

cuda_files = []

for root, dirs, files in os.walk("/kaggle/working"):
    for f in files:
        if f.endswith(".cu"):
            cuda_files.append(os.path.join(root,f))

print("Total .cu files:", len(cuda_files))

# mostrar algunos
for f in cuda_files[:10]:
    print(f)

Total .cu files: 5632
/kaggle/working/TransformerEngine/tests/cpp_distributed/test_comm_gemm.cu
/kaggle/working/TransformerEngine/tests/cpp/test_common.cu
/kaggle/working/TransformerEngine/tests/cpp/operator/test_cast.cu
/kaggle/working/TransformerEngine/tests/cpp/operator/test_cast_dbias.cu
/kaggle/working/TransformerEngine/tests/cpp/operator/test_qdq.cu
/kaggle/working/TransformerEngine/tests/cpp/operator/test_transpose.cu
/kaggle/working/TransformerEngine/tests/cpp/operator/test_act.cu
/kaggle/working/TransformerEngine/tests/cpp/operator/test_cast_gated_swiglu.cu
/kaggle/working/TransformerEngine/tests/cpp/operator/test_normalization_mxfp8.cu
/kaggle/working/TransformerEngine/tests/cpp/operator/test_cast_mxfp8_gated_swiglu.cu


In [5]:
count = 0

for file in cuda_files:

    code = open(file, errors="ignore").read()

    if "__global__" in code:
        print("Kernel encontrado en:", file)
        count += 1

        if count > 10:
            break

print("Archivos con kernels:", count)

Kernel encontrado en: /kaggle/working/TransformerEngine/transformer_engine/common/common.cu
Kernel encontrado en: /kaggle/working/TransformerEngine/transformer_engine/common/fused_softmax/scaled_aligned_causal_masked_softmax.cu
Kernel encontrado en: /kaggle/working/TransformerEngine/transformer_engine/common/fused_softmax/scaled_masked_softmax.cu
Kernel encontrado en: /kaggle/working/TransformerEngine/transformer_engine/common/fused_softmax/scaled_upper_triang_masked_softmax.cu
Kernel encontrado en: /kaggle/working/TransformerEngine/transformer_engine/common/gemm/cublaslt_grouped_gemm.cu
Kernel encontrado en: /kaggle/working/TransformerEngine/transformer_engine/common/gemm/cublaslt_gemm.cu
Kernel encontrado en: /kaggle/working/TransformerEngine/transformer_engine/common/dropout/dropout.cu
Kernel encontrado en: /kaggle/working/TransformerEngine/transformer_engine/common/fused_rope/fused_rope.cu
Kernel encontrado en: /kaggle/working/TransformerEngine/transformer_engine/common/fused_attn/

In [6]:
for file in cuda_files:

    code = open(file, errors="ignore").read()

    pos = code.find("__global__")

    if pos != -1:
        print("Archivo con kernel:", file)
        print("\n--- KERNEL PREVIEW ---\n")
        print(code[pos:pos+500])
        break

Archivo con kernel: /kaggle/working/TransformerEngine/transformer_engine/common/common.cu

--- KERNEL PREVIEW ---

__global__ void __launch_bounds__(1)
    update_tensor_scale_inv_kernel(const float *__restrict__ scale_ptr,
                                   float *__restrict__ scale_inv_ptr) {
  const float scale = scale_ptr == nullptr ? 1 : *scale_ptr;
  reciprocal<float>(scale_inv_ptr, scale);
}

}  // namespace

cudaDataType_t get_cuda_dtype(const transformer_engine::DType t) {
  using namespace transformer_engine;
  switch (t) {
    case DType::kFloat16:
      return CUDA_R_16F;
    case DType::kFloat32


In [7]:
kernel_files = []
kernel_count = 0

for file in cuda_files:

    code = open(file, errors="ignore").read()

    if "__global__" in code:
        kernel_files.append(file)
        kernel_count += code.count("__global__")

print("Archivos con kernels:", len(kernel_files))
print("Total kernels encontrados:", kernel_count)

print("\nEjemplos de archivos con kernels:")
for f in kernel_files[:10]:
    print(f)

Archivos con kernels: 1133
Total kernels encontrados: 2890

Ejemplos de archivos con kernels:
/kaggle/working/TransformerEngine/transformer_engine/common/common.cu
/kaggle/working/TransformerEngine/transformer_engine/common/fused_softmax/scaled_aligned_causal_masked_softmax.cu
/kaggle/working/TransformerEngine/transformer_engine/common/fused_softmax/scaled_masked_softmax.cu
/kaggle/working/TransformerEngine/transformer_engine/common/fused_softmax/scaled_upper_triang_masked_softmax.cu
/kaggle/working/TransformerEngine/transformer_engine/common/gemm/cublaslt_grouped_gemm.cu
/kaggle/working/TransformerEngine/transformer_engine/common/gemm/cublaslt_gemm.cu
/kaggle/working/TransformerEngine/transformer_engine/common/dropout/dropout.cu
/kaggle/working/TransformerEngine/transformer_engine/common/fused_rope/fused_rope.cu
/kaggle/working/TransformerEngine/transformer_engine/common/fused_attn/kv_cache.cu
/kaggle/working/TransformerEngine/transformer_engine/common/fused_attn/context_parallel.cu


In [8]:
count = 0
files_with_global = []

for f in cuda_files:
    code = open(f, errors="ignore").read()
    if "__global__" in code:
        count += 1
        files_with_global.append(f)

print("Archivos con __global__:", count)

print("\nEjemplos:")
for f in files_with_global[:10]:
    print(f)

Archivos con __global__: 1133

Ejemplos:
/kaggle/working/TransformerEngine/transformer_engine/common/common.cu
/kaggle/working/TransformerEngine/transformer_engine/common/fused_softmax/scaled_aligned_causal_masked_softmax.cu
/kaggle/working/TransformerEngine/transformer_engine/common/fused_softmax/scaled_masked_softmax.cu
/kaggle/working/TransformerEngine/transformer_engine/common/fused_softmax/scaled_upper_triang_masked_softmax.cu
/kaggle/working/TransformerEngine/transformer_engine/common/gemm/cublaslt_grouped_gemm.cu
/kaggle/working/TransformerEngine/transformer_engine/common/gemm/cublaslt_gemm.cu
/kaggle/working/TransformerEngine/transformer_engine/common/dropout/dropout.cu
/kaggle/working/TransformerEngine/transformer_engine/common/fused_rope/fused_rope.cu
/kaggle/working/TransformerEngine/transformer_engine/common/fused_attn/kv_cache.cu
/kaggle/working/TransformerEngine/transformer_engine/common/fused_attn/context_parallel.cu


In [9]:
device_files = []

for f in cuda_files:
    code = open(f, errors="ignore").read()

    if "__device__" in code or "__host__" in code:
        device_files.append(f)

print("Archivos con device/host:", len(device_files))

for f in device_files[:10]:
    print(f)

Archivos con device/host: 1142
/kaggle/working/TransformerEngine/transformer_engine/common/fused_softmax/scaled_aligned_causal_masked_softmax.cu
/kaggle/working/TransformerEngine/transformer_engine/common/fused_softmax/scaled_masked_softmax.cu
/kaggle/working/TransformerEngine/transformer_engine/common/fused_softmax/scaled_upper_triang_masked_softmax.cu
/kaggle/working/TransformerEngine/transformer_engine/common/gemm/cublaslt_grouped_gemm.cu
/kaggle/working/TransformerEngine/transformer_engine/common/gemm/cublaslt_gemm.cu
/kaggle/working/TransformerEngine/transformer_engine/common/dropout/dropout.cu
/kaggle/working/TransformerEngine/transformer_engine/common/fused_rope/fused_rope.cu
/kaggle/working/TransformerEngine/transformer_engine/common/fused_attn/context_parallel.cu
/kaggle/working/TransformerEngine/transformer_engine/common/fused_attn/utils.cu
/kaggle/working/TransformerEngine/transformer_engine/common/fused_attn/flash_attn.cu


## 2. Kernel Extraction & HIPIFY Integration
We use Regex-based parsing to isolate `__global__` kernels. Each kernel is then processed through `hipify-perl` (from the ROCm/HIPIFY toolchain) to create a ground-truth "Silver Dataset" of direct translations.

In [10]:
import re

kernel_pattern = re.compile(
    r"__global__\s+void\s+[a-zA-Z0-9_]+\s*\([^)]*\)\s*\{",
    re.MULTILINE
)

def extract_kernels(code):

    kernels = []

    for match in kernel_pattern.finditer(code):

        start = match.start()

        brace = 0
        i = match.end() - 1

        while i < len(code):

            if code[i] == "{":
                brace += 1

            elif code[i] == "}":
                brace -= 1

                if brace == 0:
                    kernels.append(code[start:i+1])
                    break

            i += 1

    return kernels

In [11]:
total = 0

for f in cuda_files[:100]:

    code = open(f, errors="ignore").read()

    ks = extract_kernels(code)

    if ks:
        print("archivo:", f)
        print("kernels:", len(ks))
        print("\n--- ejemplo ---\n")
        print(ks[0][:400])
        print("\n----------------\n")

    total += len(ks)

print("kernels encontrados en prueba:", total)

archivo: /kaggle/working/TransformerEngine/transformer_engine/common/fused_softmax/scaled_aligned_causal_masked_softmax.cu
kernels: 2

--- ejemplo ---

__global__ void scaled_aligned_causal_masked_softmax_warp_forward(output_t *dst, const input_t *src,
                                                                  const acc_t scale,
                                                                  const int microbatches,
                                                                  const int rows, const int cols) {
  // 1) WARP_WIDTH must 

----------------

archivo: /kaggle/working/TransformerEngine/transformer_engine/common/fused_softmax/scaled_masked_softmax.cu
kernels: 3

--- ejemplo ---

__global__ void scaled_softmax_warp_forward(output_t *dst, const input_t *src, const acc_t scale,
                                            int micro_batch_size, int element_count) {
  // WARP_SIZE and WARP_BATCH must match the return values batches_per_warp and
  // warp_size of method w

In [12]:
all_kernels = []

for f in cuda_files:

    code = open(f, errors="ignore").read()

    ks = extract_kernels(code)

    all_kernels.extend(ks)

print("Total kernels extraídos:", len(all_kernels))

Total kernels extraídos: 2448


In [13]:
import pandas as pd

df = pd.DataFrame({
    "kernel": all_kernels
})

df.to_csv("cuda_kernels_dataset.csv", index=False)

print("Dataset guardado")
print("Total kernels:", len(df))

Dataset guardado
Total kernels: 2448


In [14]:
# ============================================
# CUDA → HIP DATASET GENERATION (HIPIFY)
# ============================================
import os
import subprocess

hip_pairs = []

def run_hipify(cuda_code):
    """
    Usa la herramienta hipify-perl incluida en ROCm/HIPIFY 
    para convertir CUDA → HIP directamente.
    """
    with open("temp.cu", "w") as f:
        f.write(cuda_code)

    # Llamamos a perl y le pasamos la ruta correcta del script de HIPIFY
    result = subprocess.run(
        ["perl", "HIPIFY/bin/hipify-perl", "temp.cu"],
        capture_output=True,
        text=True
    )

    return result.stdout if result.stdout else None

for f in cuda_files:  # Limitado a 500 para evitar agotar recursos en Kaggle
    try:
        code = open(f, errors="ignore").read()

        # Solo convertimos si detectamos un kernel
        if "__global__" in code:

            hip_code = run_hipify(code)

            if hip_code:
                hip_pairs.append({
                    "input": code,
                    "output": hip_code,
                    "source": f
                })

    except Exception as e:
        print(f"No se pudo procesar {f}: {e}")
        continue

print("Pares CUDA→HIP generados:", len(hip_pairs))

Pares CUDA→HIP generados: 1133


In [15]:
import os
import subprocess
import json
from tqdm import tqdm

hip_pairs = []
archivos_procesados = 0
archivos_con_error = 0

print("Iniciando conversión de CUDA a HIP...")

# Creamos o abrimos el archivo en modo escritura (sobrescribe si existe)
with open("cuda_to_hip_dataset.jsonl", "w") as out_file:
    # Recorremos todos los archivos sin el límite [:500]
    for f in tqdm(cuda_files):
        try:
            code = open(f, errors="ignore").read()

            if "__global__" in code:
                # Archivo temporal para HIPIFY
                with open("temp.cu", "w") as temp_f:
                    temp_f.write(code)

                # Ejecutamos hipify-perl
                result = subprocess.run(
                    ["perl", "HIPIFY/bin/hipify-perl", "temp.cu"],
                    capture_output=True,
                    text=True
                )

                if result.stdout:
                    pair_data = {
                        "input": code,
                        "output": result.stdout,
                        "source": f
                    }
                    # Guardamos inmediatamente en disco para no perder datos
                    out_file.write(json.dumps(pair_data) + "\n")
                    archivos_procesados += 1

        except Exception as e:
            archivos_con_error += 1
            continue

print(f"Proceso terminado. Pares generados y guardados: {archivos_procesados}")
print(f"Errores de lectura/conversión: {archivos_con_error}")

Iniciando conversión de CUDA a HIP...


100%|██████████| 5632/5632 [02:21<00:00, 39.74it/s]

Proceso terminado. Pares generados y guardados: 1133
Errores de lectura/conversión: 0


In [16]:
# ============================================
# UPLOAD TO HUGGING FACE (SAFE TOKEN HANDLING)
# ============================================

from huggingface_hub import login, HfApi
import os
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")


login(token=HF_TOKEN)

api = HfApi()

repo_id = "Daleth-hb/cuda-hip-gpu-RAFT-dataset"

api.create_repo(repo_id=repo_id, exist_ok=True)

api.upload_file(
    path_or_fileobj="cuda_to_hip_dataset.jsonl",
    path_in_repo="cuda_to_hip_dataset.jsonl",
    repo_id=repo_id
)

print("Dataset subido a Hugging Face:", repo_id)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Dataset subido a Hugging Face: Daleth-hb/cuda-hip-gpu-RAFT-dataset


In [17]:
!pip install transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.4 MB/s eta 0:00:00


## 3. RAFT Generation (Chain-of-Thought)
In this stage, we use `Qwen2.5-Coder-7B-Instruct` as an "Oracle" to generate explanations.
* **Gold Docs:** Official HIP API signatures and descriptions.
* **Distractors:** Irrelevant C++, Python, and Web development docs to test the model's retrieval robustness.
* **Output:** A JSONL dataset where each example contains the CUDA code, the HIP code, and a **Detailed Migration Reasoning** citing specific documentation.

In [18]:
import json
import torch
import random
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import HfApi
from tqdm.auto import tqdm

# --- 1. CONFIGURACIÓN INICIAL ---
model_id = "Qwen/Qwen2.5-Coder-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="left")
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)
input_device = next(model.parameters()).device 


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [19]:
ROCM_GOLD_DOCS = [
    # --- Gestión de Memoria (Runtime) ---
    "hipMalloc: Allocates memory on the device. Equivalent to cudaMalloc. Signature: hipError_t hipMalloc(void** ptr, size_t size).",
    "hipFree: Frees device memory. Equivalent to cudaFree. Returns hipSuccess on completion.",
    "hipMemcpy: Copies data between host and device. Equivalent to cudaMemcpy. Replaces cudaMemcpyHostToDevice/DeviceToHost.",
    "hipMemcpyAsync: Asynchronous memory copy. Equivalent to cudaMemcpyAsync. Requires a hipStream_t.",
    "hipMallocManaged: Allocates Unified Memory (HSA). Equivalent to cudaMallocManaged.",
    "hipMemset: Sets device memory to a value. Equivalent to cudaMemset.",
    "hipMemsetAsync: Asynchronous version of memset. Equivalent to cudaMemsetAsync.",
    "hipHostMalloc: Allocates pinned host memory. Equivalent to cudaHostAlloc.",
    "hipHostFree: Frees pinned host memory. Equivalent to cudaFreeHost.",
    "hipHostGetDevicePointer: Gets device pointer for pinned host memory. Equivalent to cudaHostGetDevicePointer.",
    "hipMemGetInfo: Gets free and total memory on device. Equivalent to cudaMemGetInfo.",
    "hipPointerGetAttributes: Queries attributes of a pointer. Equivalent to cudaPointerGetAttributes.",

    # --- Ejecución de Kernels y Sincronización ---
    "hipLaunchKernelGGL: Macro for kernel launch. Replaces <<<g, b, s, st>>>. Args: (F, grid, block, shared, stream, args...).",
    "hipDeviceSynchronize: Blocks host until device finishes tasks. Equivalent to cudaDeviceSynchronize.",
    "hipStreamSynchronize: Blocks host until a specific stream finishes. Equivalent to cudaStreamSynchronize.",
    "__syncthreads(): Barrier synchronization for all threads in a block. Identical in HIP and CUDA.",
    "__threadfence(): Memory fence at device level. Identical in HIP and CUDA.",
    "__launch_bounds__(maxThreads, minBlocks): Kernel occupancy hint. Equivalent to cuda's __launch_bounds__.",
    "hipStreamCreate: Creates a new asynchronous stream. Equivalent to cudaStreamCreate.",
    "hipStreamCreateWithFlags: Creates stream with priority/blocking flags. Equivalent to cudaStreamCreateWithFlags.",
    "hipStreamDestroy: Deletes an asynchronous stream. Equivalent to cudaStreamDestroy.",
    "hipStreamWaitEvent: Makes a stream wait for an event. Equivalent to cudaStreamWaitEvent.",

    # --- Variables Intrínsecas (Built-in) ---
    "hipThreadIdx_x: Thread index in X dimension. Equivalent to threadIdx.x.",
    "hipThreadIdx_y: Thread index in Y dimension. Equivalent to threadIdx.y.",
    "hipThreadIdx_z: Thread index in Z dimension. Equivalent to threadIdx.z.",
    "hipBlockIdx_x: Block index in X dimension. Equivalent to blockIdx.x.",
    "hipBlockIdx_y: Block index in Y dimension. Equivalent to blockIdx.y.",
    "hipBlockIdx_z: Block index in Z dimension. Equivalent to blockIdx.z.",
    "hipBlockDim_x: Block dimensions in X. Equivalent to blockDim.x.",
    "hipGridDim_x: Grid dimensions in X. Equivalent to gridDim.x.",
    "warpSize: Number of threads in a warp (usually 64 on AMD, 32 on NVIDIA). Replaces CUDA's warpSize.",

    # --- Eventos y Tiempo ---
    "hipEventCreate: Creates an event object. Equivalent to cudaEventCreate.",
    "hipEventRecord: Records an event in a stream. Equivalent to cudaEventRecord.",
    "hipEventQuery: Queries an event's status. Equivalent to cudaEventQuery.",
    "hipEventSynchronize: Waits for an event to complete. Equivalent to cudaEventSynchronize.",
    "hipEventElapsedTime: Calculates time between two events. Equivalent to cudaEventElapsedTime.",
    "hipEventDestroy: Destroys an event. Equivalent to cudaEventDestroy.",

    # --- Gestión de Dispositivos ---
    "hipSetDevice: Sets the active device for the thread. Equivalent to cudaSetDevice.",
    "hipGetDevice: Returns the current active device. Equivalent to cudaGetDevice.",
    "hipGetDeviceCount: Returns number of available GPUs. Equivalent to cudaGetDeviceCount.",
    "hipDeviceProp_t: Struct for device properties. Equivalent to cudaDeviceProp.",
    "hipGetDeviceProperties: Fills property struct. Equivalent to cudaGetDeviceProperties.",
    "hipDeviceCanAccessPeer: Checks P2P capability. Equivalent to cudaDeviceCanAccessPeer.",
    "hipDeviceEnablePeerAccess: Enables P2P access. Equivalent to cudaDeviceEnablePeerAccess.",

    # --- Operaciones Atómicas ---
    "hipAtomicAdd: Atomic addition on memory. Equivalent to atomicAdd.",
    "hipAtomicSub: Atomic subtraction. Equivalent to atomicSub.",
    "hipAtomicExch: Atomic exchange of values. Equivalent to atomicExch.",
    "hipAtomicMin: Atomic minimum operation. Equivalent to atomicMin.",
    "hipAtomicMax: Atomic maximum operation. Equivalent to atomicMax.",
    "hipAtomicAnd: Atomic bitwise AND. Equivalent to atomicAnd.",
    "hipAtomicOr: Atomic bitwise OR. Equivalent to atomicOr.",
    "hipAtomicXor: Atomic bitwise XOR. Equivalent to atomicXor.",
    "hipAtomicCAS: Atomic Compare-And-Swap. Equivalent to atomicCAS.",

    # --- Librerías: hipBLAS (Álgebra Lineal) ---
    "hipblasCreate: Initializes hipBLAS handle. Equivalent to cublasCreate.",
    "hipblasDestroy: Releases hipBLAS resources. Equivalent to cublasDestroy.",
    "hipblasSetStream: Attaches a stream to a handle. Equivalent to cublasSetStream.",
    "hipblasHandle_t: Handle type for BLAS operations. Equivalent to cublasHandle_t.",
    "hipblasSgemm: Single-precision Matrix Multiplication. Equivalent to cublasSgemm.",
    "hipblasSaxpy: Vector addition (alpha*X + Y). Equivalent to cublasSaxpy.",
    "hipblasSdot: Vector dot product. Equivalent to cublasSdot.",
    "hipblasSnrm2: Vector L2 norm. Equivalent to cublasSnrm2.",
    "hipblasIsamax: Finds index of max element. Equivalent to cublasIsamax.",

    # --- Librerías: hipFFT (Transformada de Fourier) ---
    "hipfftPlan1d: Creates a 1D FFT plan. Equivalent to cufftPlan1d.",
    "hipfftPlan2d: Creates a 2D FFT plan. Equivalent to cufftPlan2d.",
    "hipfftExecC2C: Complex-to-Complex FFT execution. Equivalent to cufftExecC2C.",
    "hipfftExecR2C: Real-to-Complex FFT execution. Equivalent to cufftExecR2C.",
    "hipfftDestroy: Destroys an FFT plan. Equivalent to cufftDestroy.",

    # --- Librerías: hipSPARSE (Matrices Esparsas) ---
    "hipsparseCreate: Initializes sparse handle. Equivalent to cusparseCreate.",
    "hipsparseSgtsv: Solves tri-diagonal system. Equivalent to cusparseSgtsv.",
    "hipsparseCreateCsr: Creates CSR matrix descriptor. Equivalent to cusparseCreateCsr.",

    # --- Driver API (Low Level) ---
    "hipModuleLoad: Loads a code module from file. Equivalent to cuModuleLoad.",
    "hipModuleGetFunction: Gets function handle from module. Equivalent to cuModuleGetFunction.",
    "hipModuleUnload: Unloads a module. Equivalent to cuModuleUnload.",
    "hipCtxCreate: Creates a new context. Equivalent to cuCtxCreate.",
    "hipCtxDestroy: Destroys a context. Equivalent to cuCtxDestroy.",
    "hipCtxPushCurrent: Pushes context to stack. Equivalent to cuCtxPushCurrent.",
    "hipCtxPopCurrent: Pops context from stack. Equivalent to cuCtxPopCurrent.",

    # --- Funciones Matemáticas y Tipos ---
    "hipComplex: Complex number type. Equivalent to cuComplex.",
    "make_hipComplex: Constructor for complex numbers. Equivalent to make_cuComplex.",
    "hipCreal: Returns real part of complex. Equivalent to cuCreal.",
    "hipCimag: Returns imaginary part. Equivalent to cuCimag.",
    "hip_fast_sinf: Fast intrinsic sine. Equivalent to __sinf.",
    "hip_fast_expf: Fast intrinsic exponential. Equivalent to __expf.",
    "hip_fast_logf: Fast intrinsic logarithm. Equivalent to __logf.",
    "hip_fast_powf: Fast intrinsic power. Equivalent to __powf.",

    # --- Errores y Diagnóstico ---
    "hipGetLastError: Returns last error code. Equivalent to cudaGetLastError.",
    "hipPeekAtLastError: Returns last error without clearing. Equivalent to cudaPeekAtLastError.",
    "hipGetErrorString: Converts error code to string. Equivalent to cudaGetErrorString.",
    "hipGetErrorName: Returns name of the error code. Equivalent to cudaGetErrorName.",

    # --- Ocupación y Límites ---
    "hipOccupancyMaxActiveBlocksPerMultiprocessor: Calculates occupancy. Equivalent to cudaOccupancyMaxActiveBlocksPerMultiprocessor.",
    "hipDeviceGetLimit: Queries device limits (stack, heap). Equivalent to cudaDeviceGetLimit.",
    "hipDeviceSetLimit: Sets device resource limits. Equivalent to cudaDeviceSetLimit.",

    # --- Texturas (Legacy & Modern) ---
    "hipTextureObject_t: Object-based texture handle. Equivalent to cudaTextureObject_t.",
    "hipCreateTextureObject: Creates a texture object. Equivalent to cudaCreateTextureObject.",
    "hipDestroyTextureObject: Frees a texture object. Equivalent to cudaDestroyTextureObject.",
    "hipGetTextureObjectResourceDesc: Queries resource descriptor. Equivalent to cudaGetTextureObjectResourceDesc.",
    
    # --- Memoria de Símbolos ---
    "hipGetSymbolAddress: Gets device address of a symbol. Equivalent to cudaGetSymbolAddress.",
    "hipGetSymbolSize: Gets size of a symbol. Equivalent to cudaGetSymbolSize.",
    "hipMemcpyToSymbol: Copies data to device symbol. Equivalent to cudaMemcpyToSymbol.",
    "hipMemcpyFromSymbol: Copies data from device symbol. Equivalent to cudaMemcpyFromSymbol."
]

In [20]:
DISTRACTOR_DOCS = [
    # --- C++ Estándar ---
    "std::unique_ptr: Smart pointer that owns and manages another object through a pointer.",
    "std::move: Used to indicate that an object may be 'moved from'.",
    "std::map: Associative container that contains key-value pairs with unique keys.",
    "RAII: Resource Acquisition Is Initialization. A programming idiom for resource management.",
    "SFINAE: Substitution Failure Is Not An Error. A template metaprogramming technique.",
    "std::atomic: Template for objects that support atomic operations in CPU threads.",
    "std::async: Runs a function asynchronously and returns a std::future.",

    # --- Python y Scripting ---
    "Python Decorators: A way to modify the behavior of a function or class.",
    "List Comprehension: A concise way to create lists in Python.",
    "GIL (Global Interpreter Lock): A mutex that protects access to Python objects.",
    "asyncio: A library to write concurrent code using the async/await syntax.",
    "Pandas DataFrame: Two-dimensional, size-mutable, potentially heterogeneous tabular data.",
    "Numpy Broadcasting: How NumPy treats arrays with different shapes during arithmetic operations.",
    "Python Generators: Functions that return an iterable set of items, one at a time, using yield.",

    # --- Java y Enterprise ---
    "JVM Garbage Collection: Automatic memory management process in Java.",
    "Spring Boot: Framework designed to simplify the bootstrapping and development of a new Spring application.",
    "Java Interfaces: Abstract types used to specify a behavior that classes must implement.",
    "Hibernate ORM: A framework for mapping an object-oriented domain model to a relational database.",
    "Dependency Injection: A design pattern where an object receives other objects that it depends on.",

    # --- Desarrollo Web y JS ---
    "React Hooks: Functions that let you use state and other React features without writing a class.",
    "Virtual DOM: A programming concept where an ideal representation of a UI is kept in memory.",
    "JWT (JSON Web Token): A compact, URL-safe means of representing claims to be transferred between two parties.",
    "Express.js Middleware: Functions that have access to the request and response objects.",
    "Redux Store: A state container which holds the application's state tree.",
    "CSS Grid Layout: A two-dimensional grid-based layout system for the web.",
    "NPM (Node Package Manager): The default package manager for the JavaScript runtime environment Node.js.",

    # --- SQL y Bases de Datos ---
    "SQL JOIN: Clause used to combine rows from two or more tables.",
    "ACID Properties: Atomicity, Consistency, Isolation, Durability. Guaranteed database transactions.",
    "B-Tree Index: A data structure that keeps data sorted and allows searches in logarithmic time.",
    "NoSQL: Databases that provide a mechanism for storage and retrieval of data modeled in non-tabular formats.",
    "PostgreSQL WAL: Write-Ahead Logging. A standard method for ensuring data integrity.",

    # --- Cloud y DevOps ---
    "Docker Container: A standard unit of software that packages up code and all its dependencies.",
    "Kubernetes Pod: The smallest deployable units of computing that you can create and manage in Kubernetes.",
    "AWS S3: Simple Storage Service. An object storage service that offers scalability and data availability.",
    "Terraform HCL: HashiCorp Configuration Language. Used to define infrastructure as code.",
    "CI/CD Pipeline: Automated process of building, testing, and deploying code.",
    "IAM Role: An AWS identity with permission policies that determine what the identity can do in AWS.",

    # --- Redes y Seguridad ---
    "TCP Three-Way Handshake: Process used in a TCP/IP network to make a connection between server and client.",
    "TLS Handshake: The process of establishing a secure communication channel using TLS.",
    "DNS Propagation: The time it takes for DNS changes to be updated across the entire web.",
    "HTTP 404 Error: Standard response code indicating that the browser was able to communicate with a given server.",
    "REST API: Representational State Transfer. An architectural style for providing standards between computer systems.",

    # --- Inteligencia Artificial (No GPU) ---
    "Decision Trees: A non-parametric supervised learning method used for classification and regression.",
    "K-Means Clustering: A method of vector quantization that aims to partition n observations into k clusters.",
    "Backpropagation: An algorithm used in artificial neural networks to calculate a gradient that is needed in the calculation of weights.",
    "ReLU Activation: Rectified Linear Unit. A non-linear function used in neural networks.",
    "Dropout Layer: A technique where randomly selected neurons are ignored during training.",

    # --- Otros Sistemas ---
    "Bash Crontab: A system daemon used to execute scheduled programs.",
    "Git Rebase: The process of moving or combining a sequence of commits to a new base commit.",
    "SSH Protocol: Secure Shell. A cryptographic network protocol for operating network services securely.",
    "Markdown Syntax: A lightweight markup language with plain-text-formatting syntax.",
    "Kernel Panic: A safety measure taken by an operating system's kernel upon detecting an internal fatal error."
    # ... (Sigue agregando hasta completar 100 con términos como 'Linux chmod', 'JSON Parsing', 'Binary Search', etc.)
]

In [21]:
import json
import torch
import random
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import HfApi
from tqdm.auto import tqdm

# --- 3. CLASE DATASET CON LÓGICA RAFT ---
class RAFTCoTDataset(Dataset):
    def __init__(self, pairs, gold_docs, distractors):
        self.pairs = pairs
        self.gold_docs = gold_docs
        self.distractors = distractors

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair = self.pairs[idx]
        
        # Seleccionamos 3 documentos de "Oro" y 5 "Distractores" aleatorios
        relevant = random.sample(self.gold_docs, k=min(3, len(self.gold_docs)))
        irrelevant = random.sample(self.distractors, k=min(5, len(self.distractors)))
        
        context_list = relevant + irrelevant
        random.shuffle(context_list) # Mezclar para evitar sesgo de posición
        
        context_text = "\n".join([f"- {d}" for d in context_list])
        
        cuda_code = pair["input"][:2500] 
        hip_code = pair["output"][:2500]

        prompt = (
            f"### DOCUMENTATION CONTEXT:\n{context_text}\n\n"
            f"### INSTRUCTION:\nExplain the migration from CUDA to ROCm (HIP) step-by-step. "
            f"Use the provided context to justify API mappings. Ignore irrelevant documentation.\n\n"
            f"[CUDA CODE]:\n{cuda_code}\n\n"
            f"[HIP CODE]:\n{hip_code}\n\n"
            f"### DETAILED EXPLANATION:"
        )
        
        messages = [
            {"role": "system", "content": "You are a senior AMD ROCm engineer. You provide Chain-of-Thought reasoning based on official documentation."},
            {"role": "user", "content": prompt}
        ]
        
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        return {
            "text": text, 
            "cuda": pair["input"], 
            "hip": pair["output"], 
            "source": pair.get("source", "unknown")
        }

# --- 4. EJECUCIÓN PRINCIPAL ---

# Cargar datos
hip_pairs = []
with open("cuda_to_hip_dataset.jsonl", "r") as f:
    for line in f:
        hip_pairs.append(json.loads(line))

# Loader configurado para RAFT
BATCH_SIZE = 2 
loader = DataLoader(RAFTCoTDataset(hip_pairs, ROCM_GOLD_DOCS, DISTRACTOR_DOCS), batch_size=BATCH_SIZE)

print(f"Iniciando RAFT para {len(hip_pairs)} ejemplos...")

with open("cuda_hip_raft_dataset.jsonl", "w") as out_file:
    for batch in tqdm(loader):
        try:
            inputs = tokenizer(batch["text"], return_tensors="pt", padding=True, truncation=True, max_length=3500)
            inputs = {k: v.to(input_device) for k, v in inputs.items()}

            with torch.no_grad():
                generated_ids = model.generate(
                    **inputs,
                    max_new_tokens=500, 
                    pad_token_id=tokenizer.eos_token_id,
                    do_sample=False
                )
            
            prompt_lengths = inputs["input_ids"].shape[1]
            explanations = tokenizer.batch_decode(generated_ids[:, prompt_lengths:], skip_special_tokens=True)

            for i in range(len(explanations)):
                # CORRECCIÓN: f-string en una sola línea lógica para evitar SyntaxError
                record = {
                    "instruction": "Migrate this CUDA kernel to AMD ROCm (HIP) and explain the reasoning citing the provided documentation.",
                    "input": batch["cuda"][i],
                    "output": f"```cpp\n{batch['hip'][i]}\n```\n\n### Migration Reasoning:\n{explanations[i]}",
                    "source": batch["source"][i]
                }
                out_file.write(json.dumps(record) + "\n")
                
        except Exception as e:
            print(f"Error en batch: {e}")
            continue

# --- 5. SUBIDA A HUGGING FACE ---
print("Subiendo Dataset RAFT...")
api = HfApi()
repo_id = "Daleth-hb/cuda-hip-gpu-RAFT-dataset"

api.upload_file(
    path_or_fileobj="cuda_hip_raft_dataset.jsonl",
    path_in_repo="cuda_hip_raft_dataset.jsonl",
    repo_id=repo_id
)

print(f"¡Éxito total! Dataset RAFT disponible en: https://huggingface.co/datasets/{repo_id}")

Iniciando RAFT para 1133 ejemplos...


  0%|          | 0/567 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Subiendo Dataset RAFT...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

¡Éxito total! Dataset RAFT disponible en: https://huggingface.co/datasets/Daleth-hb/cuda-hip-gpu-RAFT-dataset


## 4. Dataset Statistics & Upload
Final validation of the generated pairs.
* **Total CUDA files processed:** 5,632
* **Kernels identified:** 1,133
* **Reasoning traces generated:** 1,133
* **Target Repo:** `Daleth-hb/cuda-hip-gpu-RAFT-dataset`